# Solutions – Day 30

In [ ]:
# Exercise 1: Custom preference dataset
import json
custom_data = [
    {"chosen": "I like sunny days.", "rejected": "I hate the sun."},
    {"chosen": "Cats are lovely.", "rejected": "Cats are terrible."},
]
with open("my_prefs.json", "w") as f:
    json.dump(custom_data, f)

# Load with datasets
from datasets import load_dataset
custom_ds = load_dataset("json", data_files="my_prefs.json", split="train")

In [ ]:
# Exercise 2: Use distilgpt2
from transformers import AutoModelForSequenceClassification
small_model = AutoModelForSequenceClassification.from_pretrained("distilgpt2", num_labels=1)
# Then proceed with RewardTrainer as above.

In [ ]:
# Exercise 3: Evaluation accuracy
def compute_accuracy(trainer, eval_dataset):
    correct = 0
    for batch in eval_dataset:
        chosen_score = trainer.model(batch["input_ids_chosen"]).logits
        rejected_score = trainer.model(batch["input_ids_rejected"]).logits
        if chosen_score > rejected_score:
            correct += 1
    return correct / len(eval_dataset)

In [ ]:
# Exercise 4: PPO pseudocode
ppo_psuedocode = """
policy_model = load_model("gpt2")
reward_model = load_model("./reward_model_final")
for step in range(num_steps):
    prompts = sample_prompts()
    responses = policy_model.generate(prompts)
    rewards = reward_model.score(responses)
    loss = -log_prob(policy_model, responses) * rewards  # simplified
    loss.backward()
    optimizer.step()
"""

In [ ]:
# Exercise 5: LoRA with PEFT
from peft import LoraConfig, get_peft_model
lora_config = LoraConfig(r=8, lora_alpha=32, target_modules=["c_attn"], lora_dropout=0.1)
model = AutoModelForSequenceClassification.from_pretrained("gpt2", num_labels=1)
model = get_peft_model(model, lora_config)
# Then use RewardTrainer as before